In [5]:
# ── Cell 1: Imports ───────────────────────────────────────────
import pandas as pd
import numpy as np

print("Libraries loaded successfully")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

Libraries loaded successfully
pandas version: 2.2.2
numpy version: 1.26.4


In [7]:
import os
print(os.getcwd())

C:\Users\chris


In [11]:
import os
for f in os.listdir('.'):
    if f.endswith('.csv'):
        print(f)

cleaned_diabetic_data_smaller.csv
country_comparison_large_dataset_headings (4).csv
fifa_2026_host_cities (1).csv
fifa_2026_match_schedule.csv
fifa_2026_weather_hourly_2003_2022.csv
HousePrices.csv
HousePrices_dataset.csv
penguins.csv
sf_parks.csv
whr2024-nobom.csv


In [49]:
weather = pd.read_csv('fifa_2026_weather_hourly_2003_2022.csv')
matches = pd.read_csv('fifa_2026_match_schedule.csv')
cities  = pd.read_csv('fifa_2026_host_cities (1).csv')

for name, df in [('weather', weather), ('matches', matches), ('cities', cities)]:
    print(f"{name}: {df.shape[0]:,} rows | {df.shape[1]} cols | nulls: {df.isnull().sum().sum()}")

weather: 299,520 rows | 13 cols | nulls: 0
matches: 104 rows | 4 cols | nulls: 0
cities: 16 rows | 7 cols | nulls: 0


In [51]:
# ── Cell 3: Parse dates and times ─────────────────────────────
weather['ObservationDate'] = pd.to_datetime(weather['ObservationDate'])
matches['MatchDate']       = pd.to_datetime(matches['MatchDate'])
matches['KickoffTime']     = pd.to_datetime(matches['KickoffTime'], format='%H:%M:%S').dt.time

# Extract month, day, hour from weather for joining later
weather['month'] = weather['ObservationDate'].dt.month
weather['day']   = weather['ObservationDate'].dt.day
weather['hour']  = weather['ObservationHour']

# Extract month, day, hour from matches for joining to climatology
matches['month'] = matches['MatchDate'].dt.month
matches['day']   = matches['MatchDate'].dt.day
matches['hour']  = matches['KickoffTime'].apply(lambda x: x.hour)

print("Date parsing complete")
print(f"Match date range: {matches['MatchDate'].min().date()} to {matches['MatchDate'].max().date()}")
print(f"Weather date range: {weather['ObservationDate'].min().date()} to {weather['ObservationDate'].max().date()}")
print(f"Unique kickoff hours: {sorted(matches['hour'].unique())}")

Date parsing complete
Match date range: 2026-06-11 to 2026-07-19
Weather date range: 2003-06-11 to 2022-07-19
Unique kickoff hours: [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]


In [19]:
# ── Cell 4: Calculate WBGT for every weather observation ──────
# Formula sourced from Kaggle, same as used in SQL implementation
# WBGT approximation using Steadman's apparent temperature components:
# 0.7 * wet bulb + 0.2 * globe temp + 0.1 * dry bulb

T  = weather['AirTemperature_C']
RH = weather['RelativeHumidity_pct']
SR = weather['SolarRadiation_Wm2']

wet_bulb = (
    T * np.arctan(0.151977 * np.sqrt(RH + 8.313659))
    + np.arctan(T + RH)
    - np.arctan(RH - 1.676331)
    + 0.00391838 * RH**1.5 * np.arctan(0.023101 * RH)
    - 4.686035
)

globe_temp = T + (SR / 1000) * 5

weather['WBGT'] = (0.7 * wet_bulb) + (0.2 * globe_temp) + (0.1 * T)

print(f"WBGT calculated for {len(weather):,} observations")
print(f"WBGT range: {weather['WBGT'].min():.1f}°C to {weather['WBGT'].max():.1f}°C")
print(f"Mean WBGT across all cities/times: {weather['WBGT'].mean():.1f}°C")

WBGT calculated for 299,520 observations
WBGT range: 6.2°C to 31.2°C
Mean WBGT across all cities/times: 20.2°C


In [21]:
# ── Cell 5: Build climatology (Average and 95th percentile) ───
# Group by city, month, day, hour — same logic as SQL WBGT_CLIMATOLOGY table

climatology_avg = (
    weather
    .groupby(['City', 'month', 'day', 'hour'])['WBGT']
    .mean()
    .reset_index()
    .rename(columns={'WBGT': 'WBGT_Avg'})
)

climatology_hot = (
    weather
    .groupby(['City', 'month', 'day', 'hour'])['WBGT']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'WBGT': 'WBGT_Hot'})
)

# Merge average and hot climatology into one table
climatology = climatology_avg.merge(climatology_hot, on=['City', 'month', 'day', 'hour'])

print(f"Climatology records: {len(climatology):,}")
print(f"Cities covered: {climatology['City'].nunique()}")
print(f"\nSample — Houston at 3pm on July 4th:")
sample = climatology[
    (climatology['City'] == 'Houston') &
    (climatology['month'] == 7) &
    (climatology['day'] == 4) &
    (climatology['hour'] == 15)
]
print(sample[['City', 'month', 'day', 'hour', 'WBGT_Avg', 'WBGT_Hot']].to_string(index=False))

Climatology records: 14,976
Cities covered: 16

Sample — Houston at 3pm on July 4th:
   City  month  day  hour  WBGT_Avg  WBGT_Hot
Houston      7    4    15  27.64604 29.517569


In [23]:
# ── Cell 6: Join matches to climatology and assign risk tiers ─

# First join matches to cities to get the city name
match_city = matches.merge(
    cities[['Stadium', 'City']],
    on='Stadium',
    how='left'
)

# Then join to climatology on city + month + day + hour
match_risk = match_city.merge(
    climatology,
    on=['City', 'month', 'day', 'hour'],
    how='left'
)

# Assign risk tiers — same logic as your SQL CASE statement
def assign_risk_tier(row):
    avg = row['WBGT_Avg']
    hot = row['WBGT_Hot']
    if avg > 32:
        return 5
    elif hot > 32 or avg > 28:
        return 4
    elif hot > 28 or avg > 26:
        return 3
    elif hot > 26:
        return 2
    else:
        return 1

match_risk['RiskTier'] = match_risk.apply(assign_risk_tier, axis=1)

# Sort by risk descending
match_risk = match_risk.sort_values(['RiskTier', 'WBGT_Hot'], ascending=False).reset_index(drop=True)

print(f"Matches scored: {len(match_risk)}")
print(f"\nRisk tier distribution:")
print(match_risk['RiskTier'].value_counts().sort_index())

Matches scored: 104

Risk tier distribution:
RiskTier
1    55
2    27
3    22
Name: count, dtype: int64


In [25]:
# ── Cell 7: View top risk matches ─────────────────────────────
cols = ['MatchDate', 'KickoffTime', 'Round', 'Stadium', 'City', 
        'WBGT_Avg', 'WBGT_Hot', 'RiskTier']

print("Top 15 Highest Risk Matches:")
print("=" * 75)
print(match_risk[cols].head(15).to_string(index=False))

Top 15 Highest Risk Matches:
 MatchDate KickoffTime       Round           Stadium                City  WBGT_Avg  WBGT_Hot  RiskTier
2026-07-14    14:00:00  Semifinals      AT&T Stadium              Dallas 27.890904 29.773443         3
2026-06-17    15:00:00 Group stage      AT&T Stadium              Dallas 27.384711 29.680277         3
2026-07-06    14:00:00 Round of 16      AT&T Stadium              Dallas 27.439547 29.327535         3
2026-07-03    13:00:00 Round of 32      AT&T Stadium              Dallas 27.090984 29.170509         3
2026-06-20    12:00:00 Group stage       NRG Stadium             Houston 27.386717 28.900055         3
2026-07-19    15:00:00       Final   MetLife Stadium New York/New Jersey 25.126326 28.899570         3
2026-07-04    12:00:00 Round of 16       NRG Stadium             Houston 27.138086 28.797339         3
2026-06-29    12:00:00 Round of 32       NRG Stadium             Houston 27.272612 28.735050         3
2026-06-14    15:00:00 Group stage      AT&T

In [27]:
# ── Cell 8: Export outputs for Tableau ────────────────────────

# Full match risk table
match_risk[cols].to_csv('match_risk_output.csv', index=False)

# City-level summary for map visualization
city_summary = (
    match_risk.groupby('City')
    .agg(
        TotalMatches   = ('RiskTier', 'count'),
        AvgRiskTier    = ('RiskTier', 'mean'),
        MaxRiskTier    = ('RiskTier', 'max'),
        AvgWBGT_Hot    = ('WBGT_Hot', 'mean'),
        Tier3_Matches  = ('RiskTier', lambda x: (x == 3).sum()),
        Tier2_Matches  = ('RiskTier', lambda x: (x == 2).sum()),
        Tier1_Matches  = ('RiskTier', lambda x: (x == 1).sum())
    )
    .reset_index()
    .sort_values('AvgWBGT_Hot', ascending=False)
)

city_summary.to_csv('city_risk_summary.csv', index=False)

print("Files exported successfully:")
print("  match_risk_output.csv")
print("  city_risk_summary.csv")
print(f"\nCity Risk Summary:")
print("=" * 65)
print(city_summary.to_string(index=False))

Files exported successfully:
  match_risk_output.csv
  city_risk_summary.csv

City Risk Summary:
               City  TotalMatches  AvgRiskTier  MaxRiskTier  AvgWBGT_Hot  Tier3_Matches  Tier2_Matches  Tier1_Matches
             Dallas             9     2.888889            3    28.763514              8              1              0
            Houston             7     3.000000            3    28.579647              7              0              0
              Miami             7     2.714286            3    27.573497              5              2              0
        Kansas City             6     2.166667            3    27.186252              1              5              0
            Atlanta             8     2.000000            2    26.921822              0              8              0
New York/New Jersey             8     1.750000            3    26.277366              1              4              3
       Philadelphia             6     1.666667            2    25.924621     

In [29]:
# ── Cell 9: Business Recommendations Summary ──────────────────

high_risk = match_risk[match_risk['RiskTier'] == 3][cols]
afternoon = match_risk[
    (match_risk['RiskTier'] >= 2) & 
    (match_risk['hour'] <= 15)
][cols]

print("=" * 65)
print("FIFA 2026 HEAT RISK ANALYSIS — KEY FINDINGS")
print("=" * 65)

print(f"""
TOURNAMENT OVERVIEW
-------------------
Total matches analyzed:        {len(match_risk)}
Tier 3 (High Risk):            {(match_risk['RiskTier'] == 3).sum()} matches ({(match_risk['RiskTier'] == 3).mean():.0%})
Tier 2 (Elevated Risk):        {(match_risk['RiskTier'] == 2).sum()} matches ({(match_risk['RiskTier'] == 2).mean():.0%})
Tier 1 (Low Risk):             {(match_risk['RiskTier'] == 1).sum()} matches ({(match_risk['RiskTier'] == 1).mean():.0%})

HIGHEST RISK VENUES
-------------------
1. Houston     — 100% of matches at Tier 3
2. Dallas      —  89% of matches at Tier 3
3. Miami       —  71% of matches at Tier 3

CRITICAL MATCHES
-------------------
- Semifinal (July 14, AT&T Stadium Dallas, 2pm): Highest risk match
- World Cup Final (July 19, MetLife Stadium, 3pm): Tier 3 risk
- Third Place (July 18, Hard Rock Stadium Miami, 5pm): Tier 3 risk

RECOMMENDATIONS
-------------------
1. RESCHEDULE afternoon kickoffs in Dallas and Houston to
   after 6pm — analysis shows WBGT drops significantly
   in evening hours at both venues.

2. MANDATORY cooling breaks at all Tier 3 matches per
   FIFA heat protocol guidelines.

3. PRIORITIZE retractable roof closure at AT&T Stadium
   (Dallas) and NRG Stadium (Houston) for all matches —
   both stadiums have retractable roofs per the dataset.
""")

FIFA 2026 HEAT RISK ANALYSIS — KEY FINDINGS

TOURNAMENT OVERVIEW
-------------------
Total matches analyzed:        104
Tier 3 (High Risk):            22 matches (21%)
Tier 2 (Elevated Risk):        27 matches (26%)
Tier 1 (Low Risk):             55 matches (53%)

HIGHEST RISK VENUES
-------------------
1. Houston     — 100% of matches at Tier 3
2. Dallas      —  89% of matches at Tier 3
3. Miami       —  71% of matches at Tier 3

CRITICAL MATCHES
-------------------
- Semifinal (July 14, AT&T Stadium Dallas, 2pm): Highest risk match
- World Cup Final (July 19, MetLife Stadium, 3pm): Tier 3 risk
- Third Place (July 18, Hard Rock Stadium Miami, 5pm): Tier 3 risk

RECOMMENDATIONS
-------------------
1. RESCHEDULE afternoon kickoffs in Dallas and Houston to
   after 6pm — analysis shows WBGT drops significantly
   in evening hours at both venues.

2. MANDATORY cooling breaks at all Tier 3 matches per
   FIFA heat protocol guidelines.

3. PRIORITIZE retractable roof closure at AT&T Stadium

In [31]:
import os
print(os.getcwd())

C:\Users\chris


In [33]:
match_risk[cols].to_csv('match_risk_v2.csv', index=False)
city_summary.to_csv('city_summary_v2.csv', index=False)
print("Done")

Done


In [35]:
# Export as Excel instead of CSV to avoid Tableau parsing issues
match_risk[cols].to_excel('match_risk_final.xlsx', index=False)
city_summary.to_excel('city_summary_final.xlsx', index=False)
print("Excel files exported successfully")

Excel files exported successfully


In [37]:
# Add lat/lon directly from the cities file
cities_coords = cities[['City', 'Latitude', 'Longitude']].drop_duplicates()

# Merge coordinates into match_risk
match_risk_geo = match_risk[cols].merge(cities_coords, on='City', how='left')

print(match_risk_geo[['City', 'Latitude', 'Longitude']].drop_duplicates())

# Export to Excel
match_risk_geo.to_excel('match_risk_geo.xlsx', index=False)
print("\nExported successfully")

                   City  Latitude  Longitude
0                Dallas   32.7473   -97.0945
4               Houston   29.6847   -95.4107
5   New York/New Jersey   40.8128   -74.0742
11                Miami   25.9580   -80.2389
16          Kansas City   39.0489   -94.4839
22              Atlanta   33.7555   -84.4008
24         Philadelphia   39.9008   -75.1675
34               Boston   42.0909   -71.2643
47            Monterrey   25.6701  -100.2447
55          Los Angeles   33.9535  -118.3390
57              Toronto   43.6333   -79.4186
65        San Francisco   37.4033  -121.9694
68            Vancouver   49.2768  -123.1120
70              Seattle   47.5952  -122.3316
81          Guadalajara   20.6811  -103.4627
88          Mexico City   19.3029   -99.1505

Exported successfully


In [43]:
# Create a city-level summary with max risk tier per city
city_map = (
    match_risk_geo
    .groupby(['City', 'Latitude', 'Longitude'])
    .agg(
        MaxRiskTier  = ('RiskTier', 'max'),
        TotalMatches = ('RiskTier', 'count'),
        AvgWBGT_Hot  = ('WBGT_Hot', 'mean')
    )
    .reset_index()
)

city_map.to_excel('city_map.xlsx', index=False)
print(city_map[['City', 'MaxRiskTier', 'TotalMatches']].to_string(index=False))
print("\nExported city_map.xlsx")

               City  MaxRiskTier  TotalMatches
            Atlanta            2             8
             Boston            2             7
             Dallas            3             9
        Guadalajara            1             4
            Houston            3             7
        Kansas City            3             6
        Los Angeles            1             8
        Mexico City            1             5
              Miami            3             7
          Monterrey            2             4
New York/New Jersey            3             8
       Philadelphia            2             6
      San Francisco            1             6
            Seattle            1             6
            Toronto            1             6
          Vancouver            1             7

Exported city_map.xlsx
